# Chapter 11 — Exercises: Q-Learning from scratch

**Session 4 | Chapter 11 | Exercise time: ~10 minutes (Tasks 1–4). Bonus tasks are optional.**

You implement the three building blocks of Q-Learning on the GridWorld from Ch10
(same layout as FrozenLake, deterministic — no gymnasium needed):

1. the Q-table, 2. ε-greedy action selection, 3. the TD update — and 4. plug them into a given training loop.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

rng = np.random.default_rng(42)   # one seeded generator for everything → reproducible
print('Ready!')

## Setup — run this cell (GridWorld from Ch10)

In [ ]:
# ── GridWorld: the SAME environment in Ch10, Ch11 and the animation notebook ──
# Cell types: 0 = Free, 1 = Hole, 2 = Goal, 3 = Start
# Layout is identical to Gymnasium's FrozenLake 4x4 map (SFFF / FHFH / FFFH / HFFG)
GRID = np.array([[3, 0, 0, 0],
                 [0, 1, 0, 1],
                 [0, 0, 0, 1],
                 [1, 0, 0, 2]])
ROWS, COLS = GRID.shape
N_STATES, N_ACTIONS = ROWS * COLS, 4                     # 16 states, 4 actions
ACTIONS = {0: 'Left', 1: 'Down', 2: 'Right', 3: 'Up'}   # same action coding as FrozenLake
MOVES   = {0: (0, -1), 1: (1, 0), 2: (0, 1), 3: (-1, 0)}  # (row delta, col delta)
REWARDS = {0: -0.01, 1: -1.0, 2: +1.0, 3: -0.01}         # free / hole / goal / start

def env_reset():
    """Start a new episode: the agent stands on the start cell (state 0)."""
    return 0

def env_step(state, action):
    """Apply an action. Returns (next_state, reward, done)."""
    r, c = divmod(state, COLS)
    dr, dc = MOVES[action]
    nr, nc = min(ROWS - 1, max(0, r + dr)), min(COLS - 1, max(0, c + dc))   # walls: stay put
    next_state = nr * COLS + nc
    cell = GRID[nr, nc]
    done = cell in (1, 2)                                                    # hole or goal ends the episode
    return next_state, REWARDS[cell], done

# ── Drawing helpers ──
CELL_COLORS = {0: '#eef3f5', 1: '#e74c3c', 2: '#2ecc71', 3: '#3498db'}
CELL_LABELS = {0: '', 1: 'H', 2: 'G', 3: 'S'}
ARROWS      = {0: '←', 1: '↓', 2: '→', 3: '↑'}

def draw_grid(ax, title=''):
    for r in range(ROWS):
        for c in range(COLS):
            ct = GRID[r, c]
            ax.add_patch(mpatches.FancyBboxPatch((c + 0.04, ROWS - 1 - r + 0.04), 0.92, 0.92,
                                                 boxstyle='round,pad=0.02', facecolor=CELL_COLORS[ct],
                                                 edgecolor='white', lw=2))
            if CELL_LABELS[ct]:
                ax.text(c + 0.5, ROWS - 1 - r + 0.5, CELL_LABELS[ct], ha='center', va='center',
                        fontsize=18, fontweight='bold', color='white')
            ax.text(c + 0.1, ROWS - 1 - r + 0.86, str(r * COLS + c), fontsize=8, color='gray', va='top')
    ax.set_xlim(0, COLS); ax.set_ylim(0, ROWS); ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(title, fontsize=12)

def draw_policy(ax, Q, title='Greedy policy (arrows = argmax Q)'):
    """Arrows for the best action per state; '·' where nothing was learned yet."""
    draw_grid(ax, title)
    for s in range(N_STATES):
        r, c = divmod(s, COLS)
        if GRID[r, c] in (1, 2):
            continue
        learned = np.any(Q[s] != 0)
        ax.text(c + 0.5, ROWS - r - 0.5 - (0.2 if s == 0 else 0), ARROWS[int(np.argmax(Q[s]))] if learned else '·',
                ha='center', va='center', fontsize=22, fontweight='bold', color='#1a1a1a' if learned else 'gray')

def draw_trajectory(ax, traj, color='#1a1a1a'):
    xs = [s % COLS + 0.5 for s in traj]; ys = [ROWS - s // COLS - 0.5 for s in traj]
    ax.plot(xs, ys, '-', color=color, lw=2.5, alpha=0.6)
    ax.plot(xs[-1], ys[-1], '*', color='#f39c12', ms=18, markeredgecolor='#1a1a1a')

print(f'GridWorld ready: {N_STATES} states, {N_ACTIONS} actions {ACTIONS}')

## Task 1 — The Q-table (~1 min)
One row per state, one column per action, all zeros: *"I know nothing yet."*

In [ ]:
# TODO 1: create the Q-table with shape (N_STATES, N_ACTIONS), filled with zeros
Q = None

# ▶ check
# print('Q shape:', Q.shape, '| all zeros:', np.all(Q == 0))

## Task 2 — ε-greedy action selection (~3 min)

With probability ε take a **random** action (explore), otherwise the action with the **highest Q-value** (exploit).
Use `rng.random()` for a number in [0, 1) and `rng.integers(N_ACTIONS)` for a random action.

In [ ]:
def choose_action(state, Q, epsilon):
    # TODO 2: your code here (2–3 lines)
    pass

# ▶ check
Q_test = np.tile([0.1, 0.8, 0.3, 0.2], (N_STATES, 1))   # best action everywhere = 1 (Down)
print('epsilon=0 → always exploit → should be 1:', choose_action(0, Q_test, epsilon=0.0))
print('epsilon=1 → always explore → random 0-3 :', [choose_action(0, Q_test, epsilon=1.0) for _ in range(8)])

## Task 3 — The TD update (~3 min)

```
target   = r + γ · max_a' Q(s', a')      (if the episode is done: target = r — there is no future)
Q(s, a) ← Q(s, a) + α · (target − Q(s, a))
```

In [ ]:
def q_update(Q, state, action, reward, next_state, done, alpha=0.1, gamma=0.99):
    # TODO 3: compute target and update Q[state, action] in place (3 lines)
    pass

# ▶ check
Q_test = np.zeros((N_STATES, N_ACTIONS))
q_update(Q_test, state=14, action=2, reward=1.0, next_state=15, done=True, alpha=0.1, gamma=0.99)
print('Q[14, Right] should be 0.1  (= 0 + 0.1 * (1.0 - 0)) →', Q_test[14, 2])
Q_test[15] = 0.0
q_update(Q_test, state=13, action=2, reward=-0.01, next_state=14, done=False, alpha=0.1, gamma=0.99)
print('Q[13, Right] should be 0.0089 (= 0.1 * (-0.01 + 0.99*0.1)) →', round(Q_test[13, 2], 4))

## Task 4 — Plug everything into the training loop (~3 min)

The loop is given — fill in the three marked lines.

In [ ]:
ALPHA, GAMMA = 0.1, 0.99
EPSILON, EPSILON_DECAY, EPSILON_MIN = 1.0, 0.999, 0.01
N_EPISODES, MAX_STEPS = 3000, 100

Q = np.zeros((N_STATES, N_ACTIONS))
epsilon = EPSILON
success_history = []

for episode in range(N_EPISODES):
    state = env_reset()
    for step in range(MAX_STEPS):
        action = None                                # TODO 4a: choose_action(...)
        next_state, reward, done = None, None, None  # TODO 4b: env_step(...)
        # TODO 4c: q_update(...)
        state = next_state
        if done:
            break
    success_history.append(float(done and reward > 0))
    epsilon = max(epsilon * EPSILON_DECAY, EPSILON_MIN)

print(f'Training done. Final ε = {epsilon:.3f}')
print(f'Success rate over the last 500 episodes: {np.mean(success_history[-500:]):.1%}')

In [ ]:
# ▶ Run: learning curve + learned policy (nothing to fill in)
window = 100
rolling = np.convolve(success_history, np.ones(window) / window, mode='valid')
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(rolling, color='#009090', lw=2)
axes[0].set_xlabel('episode'); axes[0].set_ylabel(f'success rate (rolling {window})'); axes[0].set_ylim(0, 1.05)
axes[0].set_title('Does your agent learn?')
draw_policy(axes[1], Q)
plt.tight_layout(); plt.show()

---
## Bonus A — Try your code on the real, slippery FrozenLake (Gymnasium)

Same map, but the agent slips. Wrap gymnasium in our `reset/step` interface and re-run *your* loop.

In [ ]:
# TODO BONUS A:
# import gymnasium as gym
# env = gym.make('FrozenLake-v1', is_slippery=True); env.reset(seed=42)
# def fl_reset():  state, _ = env.reset(); return state
# def fl_step(state, action):
#     obs, reward, terminated, truncated, _ = env.step(action)
#     return obs, reward, terminated or truncated
# → copy the loop from Task 4, replace env_reset/env_step by fl_reset/fl_step, use N_EPISODES = 5000.
# How high does the success rate get? Why not ~100%?

## Bonus B — SARSA: change ONE line

SARSA uses the Q-value of the action the agent **actually takes next** (ε-greedy!) instead of the max:

```
Q(s,a) ← Q(s,a) + α · (r + γ · Q(s', a') − Q(s,a))       # a' = choose_action(s', Q, ε)
```
Train SARSA on the GridWorld and plot both learning curves. Whose curve is lower — and what does that tell you about on-policy learning?

In [ ]:
# TODO BONUS B: implement SARSA (copy the Task 4 loop; choose next_action BEFORE the update and use Q[next_state, next_action])

---
**Solutions:** see `../04-solutions/ch11_rl_algorithms_solutions.ipynb`